# UTTA-Med — Kaggle Gate B (if needed) + Gate C (Tent + EATA)

**Settings (right sidebar):** Accelerator = **GPU T4** or **P100**. Internet = **On**.

If you uploaded `camelyon17_resnet18_source_s42.pt` as a Kaggle dataset, this skips training and only runs Tent/EATA.
If no checkpoint is found, it trains the source model (early-stop on hospital 1) then adapts.

In [ ]:
!pip -q install datasets pyarrow scikit-learn tqdm pillow
import torch, sys
print('python', sys.version.split()[0], 'torch', torch.__version__)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable GPU: Settings → Accelerator → GPU T4/P100, then Restart session'

In [ ]:
import copy, json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             precision_score, recall_score, roc_auc_score)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from datasets import load_dataset

SEED, BATCH, LR_SRC, LR_TTA, EPOCHS, PATIENCE = 42, 128, 1e-4, 1e-3, 20, 5
E_MARGIN, HF_ID = 0.4, 'wltjr1007/Camelyon17-WILDS'
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda')

ckpts = list(Path('/kaggle/input').rglob('*source_s42.pt')) + list(WORK.glob('*source_s42.pt'))
CKPT = ckpts[0] if ckpts else WORK / 'camelyon17_resnet18_source_s42.pt'
print('checkpoint path', CKPT, 'exists', Path(CKPT).exists())

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize((96, 96)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10), transforms.ColorJitter(0.1, 0.1, 0.1, 0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.Resize((96, 96)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def cid(v):
    return int(v.item() if hasattr(v, 'item') else v)

class HFCamelyon(Dataset):
    def __init__(self, hf_split, centers, transform):
        self.transform = transform
        centers = {int(c) for c in centers}
        keep = [i for i, c in enumerate(hf_split['center']) if cid(c) in centers]
        self.ds = hf_split.select(keep) if len(keep) < len(hf_split) else hf_split
        print(f'  kept {len(self.ds)}/{len(hf_split)} centers={sorted(centers)}')
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        row = self.ds[i]; img = row['image']
        img = img.convert('RGB') if isinstance(img, Image.Image) else Image.fromarray(np.array(img)).convert('RGB')
        y = float(int(row['label']))
        return self.transform(img), torch.tensor(y, dtype=torch.float32)

class ResNet18Classifier(nn.Module):
    def __init__(self, dropout_p=0.5, pretrained=False):
        super().__init__()
        w = models.ResNet18_Weights.DEFAULT if pretrained else None
        bb = models.resnet18(weights=w)
        dim = bb.fc.in_features; bb.fc = nn.Identity(); self.backbone = bb
        self.classifier = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(inplace=True), nn.Dropout(dropout_p), nn.Linear(dim, 1))
    def forward(self, x): return self.classifier(self.backbone(x))

print('Loading', HF_ID, '(~10.7GB first time, then cached)')
raw = load_dataset(HF_ID)
train_set = HFCamelyon(raw['train'], {0, 3, 4}, train_tf)
id_set    = HFCamelyon(raw['validation'], {0, 3, 4}, eval_tf)
ood_set   = HFCamelyon(raw['validation'], {1}, eval_tf)
test_set  = HFCamelyon(raw['test'], {2}, eval_tf)

def loader(ds, shuffle, bs=BATCH):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader, id_loader, ood_loader, test_loader = loader(train_set, True), loader(id_set, False), loader(ood_set, False), loader(test_set, False, 64)
print('n', len(train_set), len(id_set), len(ood_set), len(test_set))

In [ ]:
def ece_score(y, p, n_bins=15):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (p > bins[i]) & (p <= bins[i+1]) if i else (p >= bins[i]) & (p <= bins[i+1])
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)

def metrics_of(y, p):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel(); pred = (p >= 0.5).astype(int)
    return {'n': int(len(y)), 'accuracy': float(accuracy_score(y, pred)),
            'precision': float(precision_score(y, pred, zero_division=0)),
            'recall': float(recall_score(y, pred, zero_division=0)),
            'sensitivity': float(recall_score(y, pred, zero_division=0)),
            'specificity': float(((y==0)&(pred==0)).sum() / max((y==0).sum(), 1)),
            'f1': float(f1_score(y, pred, zero_division=0)),
            'auroc': float(roc_auc_score(y, p)), 'auprc': float(average_precision_score(y, p)),
            'ece': ece_score(y, p), 'brier': float(np.mean((p-y)**2))}

def flip_table(y, before, after):
    y = np.asarray(y).ravel().astype(int)
    b = (np.asarray(before).ravel() >= 0.5).astype(int); a = (np.asarray(after).ravel() >= 0.5).astype(int)
    return {'stable': int(((b==y)&(a==y)).sum()), 'correction': int(((b!=y)&(a==y)).sum()),
            'persistent': int(((b!=y)&(a!=y)).sum()), 'harmful': int(((b==y)&(a!=y)).sum()),
            'correction_rate': float(((b!=y)&(a==y)).sum() / max((b!=y).sum(), 1)),
            'harm_rate': float(((b==y)&(a!=y)).sum() / max((b==y).sum(), 1)),
            'flip_rate': float((a!=b).mean())}

@torch.no_grad()
def evaluate(model, ld):
    model.eval(); ys, ps = [], []
    for x, y in ld:
        x = x.to(device, non_blocking=True)
        ps.append(torch.sigmoid(model(x)).view(-1).cpu().numpy()); ys.append(y.view(-1).numpy())
    return metrics_of(np.concatenate(ys), np.concatenate(ps))

In [ ]:
def train_source():
    model = ResNet18Classifier(pretrained=True).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR_SRC, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(); scaler = torch.amp.GradScaler('cuda')
    best, stale, history = -1.0, 0, []
    for epoch in range(1, EPOCHS+1):
        model.train(); run = n = 0
        pbar = tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}')
        for x, y in pbar:
            x, y = x.to(device, non_blocking=True), y.view(-1,1).to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()*x.size(0); n += x.size(0); pbar.set_postfix(loss=f'{loss.item():.4f}')
        val = evaluate(model, ood_loader)
        history.append({'epoch': epoch, 'train_loss': run/max(n,1), **{f'val_{k}': v for k,v in val.items()}})
        print(f"epoch {epoch:02d} loss={run/max(n,1):.4f} val_auroc={val['auroc']:.4f} val_f1={val['f1']:.4f} val_ece={val['ece']:.4f}")
        if val['auroc'] > best:
            best, stale = val['auroc'], 0
            torch.save({'model': {k:v.detach().cpu() for k,v in model.state_dict().items()},
                        'epoch': epoch, 'val_auroc': best, 'seed': SEED}, CKPT)
            print('  saved', CKPT)
        else:
            stale += 1
            if stale >= PATIENCE:
                print('early stop'); break
    return history

history = []
if Path(CKPT).exists():
    print('Using existing checkpoint — skip source training')
else:
    print('No checkpoint — running Gate B-FULL source training')
    history = train_source()

In [ ]:
def load_fresh():
    m = ResNet18Classifier(pretrained=False).to(device)
    st = torch.load(CKPT, map_location='cpu', weights_only=False)
    m.load_state_dict(st['model']); return m, st

model, st = load_fresh()
print('loaded epoch', st.get('epoch'), 'val_auroc', st.get('val_auroc'))
id_m, ood_m, test_m = evaluate(model, id_loader), evaluate(model, ood_loader), evaluate(model, test_loader)
source_json = {
    'cuda': True, 'device': torch.cuda.get_device_name(0),
    'checkpoint': str(CKPT), 'best_epoch': st.get('epoch'),
    'id_val': id_m, 'val_ood': ood_m, 'test_target': test_m,
    'shift': {'auroc_drop': id_m['auroc']-test_m['auroc'], 'auprc_drop': id_m['auprc']-test_m['auprc'],
              'f1_drop': id_m['f1']-test_m['f1'], 'acc_drop': id_m['accuracy']-test_m['accuracy']},
    'history': history,
}
(WORK/'camelyon17_resnet18_source_s42.json').write_text(json.dumps(source_json, indent=2))
print('SOURCE  id', round(id_m['auroc'],4), 'ood', round(ood_m['auroc'],4), 'test', round(test_m['auroc'],4),
      'F1 drop', round(source_json['shift']['f1_drop'],4))

In [ ]:
def collect_bn_affine(m):
    for p in m.parameters(): p.requires_grad = False
    params = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d) and mod.affine:
            mod.weight.requires_grad = True; mod.bias.requires_grad = True; params += [mod.weight, mod.bias]
    assert params; return params

def tta_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.Dropout): mod.eval()
        elif isinstance(mod, nn.BatchNorm2d): mod.train()

def binary_entropy(logits):
    p = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
    return -(p*torch.log(p)+(1-p)*torch.log(1-p)).view(-1)

def adapt_stream(method, e_margin=E_MARGIN):
    m, _ = load_fresh(); src, _ = load_fresh(); src.eval()
    opt = torch.optim.Adam(collect_bn_affine(m), lr=LR_TTA)
    n_seen = n_kept = n_skip = 0; ys, pb, pa = [], [], []
    t0 = time.time()
    for x, y in tqdm(test_loader, desc=method):
        x = x.to(device, non_blocking=True)
        with torch.no_grad(): pb.append(torch.sigmoid(src(x)).view(-1).cpu().numpy())
        tta_mode(m); n_seen += int(x.size(0))
        if method == 'tent':
            opt.zero_grad(set_to_none=True); binary_entropy(m(x)).mean().backward(); opt.step(); n_kept += int(x.size(0))
        elif method == 'eata':
            with torch.no_grad(): keep = binary_entropy(m(x)) < e_margin
            k = int(keep.sum().item())
            if k == 0: n_skip += 1
            else:
                opt.zero_grad(set_to_none=True); binary_entropy(m(x))[keep].mean().backward(); opt.step(); n_kept += k
        with torch.no_grad():
            m.eval(); pa.append(torch.sigmoid(m(x)).view(-1).cpu().numpy())
        ys.append(y.view(-1).numpy())
    y, b, a = np.concatenate(ys), np.concatenate(pb), np.concatenate(pa)
    return {'method': method, 'protocol': 'continual_one_pass_no_shuffle_bn_affine',
            'seconds': round(time.time()-t0,1), 'n_seen': n_seen, 'n_adapted': n_kept,
            'coverage': n_kept/max(n_seen,1), 'n_skip_batches': n_skip,
            'before': metrics_of(y,b), 'after': metrics_of(y,a), 'harm': flip_table(y,b,a)}

tent = adapt_stream('tent')
eata = adapt_stream('eata')
out = {'cuda': True, 'device': torch.cuda.get_device_name(0), 'checkpoint': str(CKPT),
       'source_test': test_m, 'tent_test': tent, 'eata_test': eata}
p = WORK/'camelyon17_resnet18_tent_eata_s42.json'
p.write_text(json.dumps(out, indent=2))
print('\n=== GATE C ===')
for name, r in [('source', {'after': test_m, 'coverage': 0, 'harm': {'harm_rate': None}}), ('tent', tent), ('eata', eata)]:
    print(f"{name:7s} AUROC={r['after']['auroc']:.4f} F1={r['after']['f1']:.4f} ECE={r['after']['ece']:.4f} cov={r.get('coverage',0):.3f} harm={r.get('harm',{}).get('harm_rate')}")
print('wrote', p)
print('\nDownload from /kaggle/working: camelyon17_resnet18_source_s42.json and camelyon17_resnet18_tent_eata_s42.json')